In [1]:
!pip install wbdata plotly --quiet

import wbdata
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import datetime
import warnings
warnings.filterwarnings('ignore')

print("✅ Libraries loaded")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.4/300.4 kB 2.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
moviepy 1.0.3 requires decorator<5.0,>=4.0.2, but you have decorator 5.2.1 which is incompatible.
✅ Libraries loaded


In [2]:
paises = ["AR", "DE"]
nombres = {"AR": "Argentina 🇦🇷", "DE": "Germany 🇩🇪"}

# World Bank Indicators
indicadores = {
    "FP.CPI.TOTL.ZG": "Inflation (%)",
    "NY.GDP.PCAP.CD": "GDP per Capita (USD)",
    "SL.UEM.TOTL.ZS": "Unemployment (%)",
    "GC.DOD.TOTL.GD.ZS": "Government Debt (% of GDP)",
    "NY.GDP.MKTP.KD.ZG": "GDP Growth (%)"
}

# Period: the last 20 years
fecha_inicio = datetime.datetime(2004, 1, 1)
fecha_fin = datetime.datetime(2023, 1, 1)

# Download all indicators
dfs = {}
for codigo, nombre in indicadores.items():
    try:
        df = wbdata.get_dataframe(
            {codigo: nombre},
            country=paises,
            date=(fecha_inicio, fecha_fin)
        )
        dfs[nombre] = df.unstack(level=0)
        print(f"✅ {nombre}")
    except Exception as e:
        print(f"❌ {nombre}: {e}")

print("\n✅ Data downloaded successfully")

✅ Inflation (%)
✅ GDP per Capita (USD)
✅ Unemployment (%)
✅ Government Debt (% of GDP)
✅ GDP Growth (%)

✅ Data downloaded successfully


In [4]:
inflacion = dfs["Inflation (%)"].copy()

# Create a mapping from country names (as they appear in the unstacked DataFrame columns)
# to their full names with emojis.
country_name_to_full_emoji = {
    "Argentina": nombres["AR"],
    "Germany": nombres["DE"]
}

# The columns are a MultiIndex like (indicator, country_name).
# We want to replace the 'country_name' part with the full name from `country_name_to_full_emoji`
# and flatten the columns to a single level.

# Extract the second level of the MultiIndex (which contains the country names like 'Argentina', 'Germany')
original_country_names = [col_tuple[1] for col_tuple in inflacion.columns]

# Create new column names by mapping the original country names to their full names with emojis
new_column_names = [country_name_to_full_emoji[name] for name in original_country_names]

# Assign the new, single-level column names to the DataFrame
inflacion.columns = new_column_names

inflacion.index = pd.to_datetime(inflacion.index)
inflacion = inflacion.sort_index()

print("📊 Inflation data preview:")
print(inflacion.tail(10))

# Now, these column names will be correctly accessible
print(f"\nArgentina max inflation: {inflacion['Argentina 🇦🇷'].max():.1f}%")
print(f"Germany max inflation: {inflacion['Germany 🇩🇪'].max():.1f}%")

📊 Inflation data preview:
            Argentina 🇦🇷  Germany 🇩🇪
date                                
2014-01-01           NaN    0.906794
2015-01-01           NaN    0.514426
2016-01-01           NaN    0.491747
2017-01-01           NaN    1.509495
2018-01-01     34.277224    1.732169
2019-01-01     53.548304    1.445660
2020-01-01     42.015095    0.144878
2021-01-01     48.409379    3.066667
2022-01-01     72.430758    6.872574
2023-01-01    133.488936    5.946437

Argentina max inflation: 133.5%
Germany max inflation: 6.9%


In [6]:
inflacion = dfs["Inflation (%)"].copy()

# Create a mapping from country names (as they appear in the unstacked DataFrame columns)
# to their full names with emojis.
country_name_to_full_emoji = {
    "Argentina": nombres["AR"],
    "Germany": nombres["DE"]
}

# The columns are a MultiIndex like (indicator, country_name).
# We want to replace the 'country_name' part with the full name from `country_name_to_full_emoji`
# and flatten the columns to a single level.

# Extract the second level of the MultiIndex (which contains the country names like 'Argentina', 'Germany')
original_country_names = [col_tuple[1] for col_tuple in inflacion.columns]

# Create new column names by mapping the original country names to their full names with emojis
new_column_names = [country_name_to_full_emoji[name] for name in original_country_names]

# Assign the new, single-level column names to the DataFrame
inflacion.columns = new_column_names

inflacion.index = pd.to_datetime(inflacion.index)
inflacion = inflacion.sort_index()

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=inflacion.index,
    y=inflacion["Argentina 🇦🇷"],
    name="Argentina 🇦🇷",
    line=dict(color="#74ACDF", width=3),
    fill='tozeroy',
    fillcolor='rgba(116, 172, 223, 0.15)'
))

fig.add_trace(go.Scatter(
    x=inflacion.index,
    y=inflacion["Germany 🇩🇪"],
    name="Germany 🇩🇪",
    line=dict(color="#000000", width=3),
))

# Línea de referencia al 10%
fig.add_hline(y=10, line_dash="dash", line_color="red",
              annotation_text="10% threshold", annotation_position="right")

fig.update_layout(
    title=dict(
        text="Inflation Rate (%) — Argentina vs Germany (2004–2023)",
        font=dict(size=20), x=0.5
    ),
    yaxis_title="Inflation (%)",
    xaxis_title="Year",
    template="plotly_white",
    height=500,
    legend=dict(orientation='h', y=1.05),
    annotations=[dict(
        x=0.01, y=0.97, xref='paper', yref='paper',
        text="⚠️ Argentina exceeded 100% inflation in 2023",
        showarrow=False, font=dict(size=12, color="red"),
        bgcolor="rgba(255,200,200,0.8)"
    )]
)

fig.write_html("inflation_comparison.html", include_plotlyjs='cdn')
fig.show()
print("✅ Saved")

✅ Saved


In [8]:
gdp = dfs["GDP per Capita (USD)"].copy()

# Create a mapping from country names (as they appear in the unstacked DataFrame columns)
# to their full names with emojis.
country_name_to_full_emoji = {
    "Argentina": nombres["AR"],
    "Germany": nombres["DE"]
}

# The columns are a MultiIndex like (indicator, country_name).
# We want to replace the 'country_name' part with the full name from `country_name_to_full_emoji`
# and flatten the columns to a single level.

# Extract the second level of the MultiIndex (which contains the country names like 'Argentina', 'Germany')
original_country_names = [col_tuple[1] for col_tuple in gdp.columns]

# Create new column names by mapping the original country names to their full names with emojis
new_column_names = [country_name_to_full_emoji[name] for name in original_country_names]

# Assign the new, single-level column names to the DataFrame
gdp.columns = new_column_names

gdp.index = pd.to_datetime(gdp.index)
gdp = gdp.sort_index()

fig = make_subplots(specs=[[{"secondary_y": True}]])

fig.add_trace(go.Bar(
    x=gdp.index, y=gdp["Germany 🇩🇪"],
    name="Germany 🇩🇪",
    marker_color='rgba(0,0,0,0.7)'
), secondary_y=False)

fig.add_trace(go.Bar(
    x=gdp.index, y=gdp["Argentina 🇦🇷"],
    name="Argentina 🇦🇷",
    marker_color='rgba(116, 172, 223, 0.8)'
), secondary_y=False)

fig.update_layout(
    title=dict(text="GDP per Capita (USD) — Argentina vs Germany (2004–2023)",
               font=dict(size=20), x=0.5),
    template="plotly_white",
    height=500,
    barmode='group',
    legend=dict(orientation='h', y=1.05)
)
fig.update_yaxes(title_text="GDP per Capita (USD)")

fig.write_html("gdp_comparison.html", include_plotlyjs='cdn')
fig.show()
print("✅ Saved")

✅ Saved


In [10]:
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        "Inflation (%)",
        "GDP per Capita (USD)",
        "Unemployment (%)",
        "GDP Growth (%)"
    ),
    vertical_spacing=0.15,
    horizontal_spacing=0.1
)

colores = {"Argentina 🇦🇷": "#74ACDF", "Germany 🇩🇪": "#000000"}
indicadores_plot = [
    ("Inflation (%)", 1, 1),
    ("GDP per Capita (USD)", 1, 2),
    ("Unemployment (%)", 2, 1),
    ("GDP Growth (%)", 2, 2)
]

for ind_nombre, row, col in indicadores_plot:
    if ind_nombre not in dfs:
        continue
    df_ind = dfs[ind_nombre].copy()

    # Create a mapping from country names (as they appear in the unstacked DataFrame columns)
    # to their full names with emojis.
    country_name_to_full_emoji = {
        "Argentina": nombres["AR"],
        "Germany": nombres["DE"]
    }

    # The columns are a MultiIndex like (indicator, country_name).
    # We want to replace the 'country_name' part with the full name from `country_name_to_full_emoji`
    # and flatten the columns to a single level.

    # Extract the second level of the MultiIndex (which contains the country names like 'Argentina', 'Germany')
    original_country_names = [col_tuple[1] for col_tuple in df_ind.columns]

    # Create new column names by mapping the original country names to their full names with emojis
    new_column_names = [country_name_to_full_emoji[name] for name in original_country_names]

    # Assign the new, single-level column names to the DataFrame
    df_ind.columns = new_column_names

    df_ind.index = pd.to_datetime(df_ind.index)
    df_ind = df_ind.sort_index()

    for pais in ["Argentina 🇦🇷", "Germany 🇩🇪"]:
        if pais in df_ind.columns:
            fig.add_trace(go.Scatter(
                x=df_ind.index,
                y=df_ind[pais],
                name=pais,
                line=dict(color=colores[pais], width=2),
                showlegend=(row == 1 and col == 1)
            ), row=row, col=col)

fig.update_layout(
    title=dict(
        text="Macroeconomic Dashboard — Argentina 🇦🇷 vs Germany 🇩🇪 (2004–2023)",
        font=dict(size=18), x=0.5
    ),
    height=700,
    template="plotly_white",
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1)
)

fig.write_html("macro_dashboard.html", include_plotlyjs='cdn')
fig.show()
print("✅ Main dashboard saved")

✅ Main dashboard saved


In [12]:
resumen_data = []
for ind_nombre in ["Inflation (%)", "GDP per Capita (USD)",
                    "Unemployment (%)", "GDP Growth (%)"]:
    if ind_nombre not in dfs:
        continue
    df_ind = dfs[ind_nombre].copy()

    # Create a mapping from country names (as they appear in the unstacked DataFrame columns)
    # to their full names with emojis.
    country_name_to_full_emoji = {
        "Argentina": nombres["AR"],
        "Germany": nombres["DE"]
    }

    # The columns are a MultiIndex like (indicator, country_name).
    # We want to replace the 'country_name' part with the full name from `country_name_to_full_emoji`
    # and flatten the columns to a single level.

    # Extract the second level of the MultiIndex (which contains the country names like 'Argentina', 'Germany')
    original_country_names = [col_tuple[1] for col_tuple in df_ind.columns]

    # Create new column names by mapping the original country names to their full names with emojis
    new_column_names = [country_name_to_full_emoji[name] for name in original_country_names]

    # Assign the new, single-level column names to the DataFrame
    df_ind.columns = new_column_names

    df_ind.index = pd.to_datetime(df_ind.index)

    for pais in ["Argentina 🇦🇷", "Germany 🇩🇪"]:
        if pais in df_ind.columns:
            serie = df_ind[pais].dropna()
            if not serie.empty: # Only append if there's actual data after dropping NaNs
                resumen_data.append({
                    "Indicator": ind_nombre,
                    "Country": pais,
                    "Average (2004-2023)": round(serie.mean(), 2),
                    "Max": round(serie.max(), 2),
                    "Min": round(serie.min(), 2),
                    "Latest (2023)": round(serie.iloc[-1], 2)
                })

resumen = pd.DataFrame(resumen_data)
print("📊 SUMMARY TABLE")
print(resumen.to_string(index=False))

📊 SUMMARY TABLE
           Indicator      Country  Average (2004-2023)      Max      Min  Latest (2023)
       Inflation (%) Argentina 🇦🇷                64.03   133.49    34.28         133.49
       Inflation (%)   Germany 🇩🇪                 1.97     6.87     0.14           5.95
GDP per Capita (USD) Argentina 🇦🇷             10534.41 14532.50  4242.02       14261.85
GDP per Capita (USD)   Germany 🇩🇪             45038.91 54776.77 34566.74       54776.77
    Unemployment (%) Argentina 🇦🇷                 8.64    13.52     6.14           6.14
    Unemployment (%)   Germany 🇩🇪                 5.89    11.19     3.07           3.07
      GDP Growth (%) Argentina 🇦🇷                 2.58    10.44    -9.90          -1.86
      GDP Growth (%)   Germany 🇩🇪                 1.23     4.13    -5.54          -0.87
